In [0]:
from typing import Any

In [0]:
import logging

from py4j.java_gateway import JavaObject  # type: ignore
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    coalesce,
    concat_ws,
    current_timestamp,
    from_utc_timestamp,
    lit,
    sha2,
    substring,
    when,
)

In [0]:
dbutils.widgets.text('curated_zone', 'None', '01-Curated_zone')
dbutils.widgets.text('catalog_landed', 'None', '02-Catalog_landed')
dbutils.widgets.text('reference_zone', 'None', '03-Reference_zone')
dbutils.widgets.text('run_date', 'None', '04-Run Date')
dbutils.widgets.text('schema', 'None', '05-Schema')
dbutils.widgets.text('env_prefix', 'None', '06-Environment')
curated_zone = dbutils.widgets.get('curated_zone')
catalog_landed = dbutils.widgets.get('catalog_landed')
reference_zone = dbutils.widgets.get('reference_zone')
run_date = dbutils.widgets.get('run_date')
schema = dbutils.widgets.get('schema')
env_prefix = dbutils.widgets.get('env_prefix')

import sys, os
relative_path = '../../helpers/lng_lnd'
path = os.path.abspath(relative_path)
if path not in sys.path:
  sys.path.append(path)
from landing_helpers import *
from landed_helpers import *
relative_path = '../../helpers/curated_helpers'
path = os.path.abspath(relative_path)
if path not in sys.path:
  sys.path.append(path)
from delta_table_creator import *
from extract import *
from transform import *
spark = SparkSession.builder.getOrCreate()


In [0]:
def load_curated_party_qca(
    spark, curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix
) -> None:
    args = [curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix]
    if all(item is not None for item in args) :
        date_filter = f"businessdate = {run_date!r}"

        # schema = (
        #     "PartyID string"
        #     ",SourceSystem string"
        #     ",PortfolioKey string"
        #     ",PortfolioName string"
        #     ",IngestionTimestamp timestamp"
        #     ",BatchId string"
        #     ",CDI string"
        #     ",RepresentedByPartyKey string"
        #     ",RepresentedByPartyName string"
        #     ",RepresentedByPartyType string"
        #     ",PartyKey string"
        #     ",PartyRepresentKey string"
        #     ",PartyTypeCode string"
        #     ",PartyType string"
        #     ",PartyStatusCode string"
        #     ",PartyStatus string"
        #     ",PartyStatusDate date"
        #     ",L3869StatusCode string"
        #     ",L3869Status string"
        #     ",L3869CourtDate timestamp"
        #     ",OCWStatus string"
        #     ",DeceasedFlag boolean"
        #     ",L3869InterbankingStatusCode int"
        #     ",L3869InterBankingStatus string"
        #     ",Watchlist string"
        #     ",FirstNameGR string"
        #     ",FirstNameENG string"
        #     ",LastNameGR string"
        #     ",LastNameENG string"
        #     ",FirstFatherNameGR string"
        #     ",FirstFatherNameENG string"
        #     ",LastFatherNameGR string"
        #     ",LastFatherNameENG string"
        #     ",FirstMotherNameGR string"
        #     ",LastMotherNameGR string"
        #     ",LastMotherNameENG string"
        #     ",LegalNameGR string"
        #     ",LegalNameENG string"
        #     ",PrimaryCommercialGR string"
        #     ",PrimaryCommercialENG string"
        #     ",SalutationTypeCode string"
        #     ",SalutationType string"
        #     ",GenderCode string"
        #     ",Gender string"
        #     ",MaritalStatusCode string"
        #     ",MaritalStatus string"
        #     ",CitizenshipCode string"
        #     ",Citizenship string"
        #     ",NationalityCode string"
        #     ",Nationality string"
        #     ",OccupationCode string"
        #     ",Occupation string"
        #     ",BirthDate string"
        #     ",BirthPlace string"
        #     ",DeathDate date"
        #     ",CommunicationLanguageCode int"
        #     ",CommunicationLanguage string"
        #     ",TaxRegistrationNumber string"
        #     ",TaxResidenceCountry string"
        #     ",OrganizationOriginCountry string"
        #     ",PassportNumber string"
        #     ",PassportIssuingAuthority string"
        #     ",PassportIssueDate date"
        #     ",PassportExpiredDate date"
        #     ",NationalNumber string"
        #     ",NationalNumberIssuingAuthority string"
        #     ",NationalNumberIssueDate date"
        #     ",NationalNumberExpiredDate date"
        #     ",MilitaryNumber string"
        #     ",MilitaryAuthority string"
        #     ",MilitaryIssueDate date"
        #     ",MilitaryExpiredDate date"
        #     ",SocialSecurityNumber string"
        #     ",ChildrenCertification string"
        #     ",ChildrenCertificationIssuingAuthority string"
        #     ",EnterpriseStartDate string"
        #     ",EnterpriseEndDate date"
        #     ",EconomicActivityCode int"
        #     ",EconomicActivity string"
        #     ",BranchActivityCode int"
        #     ",BranchActivity string"
        #     ",NumberOfChildren string"
        #     ",GeneralCommercialRegister string"
        #     ",NumberofEmployees string"
        #     ",TotalIncome string"
        #     ",BalanceSheetTotalAmount string"
        #     ",AnnualTurnoverAmount string"
        #     ",GroupCode string"
        #     ",GroupName string"
        #     ",GroupCodePortfolio int"
        #     ",PartyBankruptcyStage int"
        #     ",CovidFlag string"
        #     ",CovidCategoryCode int"
        #     ",CovidCategory string"
        #     ",AffectedCategoryCode int"
        #     ",AffectedCategory string"
        #     ",MonitoringUnitCode string"
        #     ",MonitoringUnit string"
        #     ",CreditRatingTypeCode int"
        #     ",CreditRatingType string"
        #     ",EducationLevelCode int"
        #     ",EducationLevel string"
        #     ",BalanceCategoryCode string"
        #     ",BalanceCategory string"
        #     ",ExtraSegmentCode string"
        #     ",ExtraSegment string"
        #     ",SegmentCode string"
        #     ",Segment string"
        #     ",SubSegmentCode string"
        #     ",SubSegment string"
        #     ",LegalFormCode int"
        #     ",LegalForm string"
        #     ",IsWholeshaleFlag boolean"
        #     ",EmploymentStatusCode int"
        #     ",EmploymentStatus string"
        #     ",EmploymentStatusDate date"
        #     ",CreatedBy string"
        #     ",CreatedByDate timestamp"
        #     ",LastUpdatedBy string"
        #     ",LastUpdatedByDate timestamp"
        #     ",ResubmissionStatus string"
        #     ",N4605Flag int"
        #     ",ConnectedBorrowerFlag boolean"
        #     ",StaffFlag string"
        #     ",L3869BorrowerHasOver50KBalance boolean"
        #     ",TargetLegalActionCode int"
        #     ",TargetLegalAction string"
        #     ",SubServicer string"
        #     ",RiskDescription string"
        #     ",MobileTel string"
        #     ",MobileTel2 string"
        #     ",BusinessDate date"
        # )
        cluster_columns = ["BusinessDate"]

        curated_table_path = f"{curated_zone}.Party"

        # ================== Create Delta Table ==================
        logging.info("Creating Delta Table...")
        if spark.catalog.tableExists(curated_table_path) is False:
            create_delta_table(
                spark,
                curated_table_path,
                cluster_columns,
                schema,  # type: ignore
            )
        else:
            logging.info("Table already exists.")

        # ================== Delete From Delta Table ==================
        if spark.catalog.tableExists(curated_table_path) is True:
            delete_from_curated_table(
                spark,
                curated_table_path,
                run_date,  # type: ignore
                "qca",
            )        
        else:
            logging.info("Tables does not exist, no need to delete entries for business date!")

        # ============================== fsad_customers_extra_open ====================

        fsad_customers_extra_open_extra_filters = None

        if fsad_customers_extra_open_extra_filters is not None:
            fsad_customers_extra_open_extra_filters.append(date_filter)
            fsad_customers_extra_open_final_filters = " AND ".join(
                fsad_customers_extra_open_extra_filters
            )
        else:
            fsad_customers_extra_open_final_filters = date_filter

        fsad_customers_extra_open_selected_fields = [
            "cuex_generic_text_02",
            "cust_code",
        ]

        fsad_customers_extra_open = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "qca_fsad_customers_extra_open",
            fsad_customers_extra_open_final_filters,
            fsad_customers_extra_open_selected_fields,
        )

        # ==============================fs_customers_snap_extra====================

        fs_customers_snap_extra_extra_filters = None

        if fs_customers_snap_extra_extra_filters is not None:
            fs_customers_snap_extra_extra_filters.append(date_filter)
            fs_customers_snap_extra_final_filters = " AND ".join(
                fs_customers_snap_extra_extra_filters
            )
        else:
            fs_customers_snap_extra_final_filters = date_filter

        fs_customers_snap_extra_selected_fields = [
            "cust_code",
            "cusx_generic_bit_01",
            "CUSX_GENERIC_NVARCHAR_01",
            "CUSX_GENERIC_BIT_02",
            "CUSX_GENERIC_BIT_05",
        ]

        fs_customers_snap_extra = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "qca_fs_customers_snap_extra",
            fs_customers_snap_extra_final_filters,
            fs_customers_snap_extra_selected_fields,
        )

        # =============================fs_customer_snap=========================
        fs_customers_snap_extra_filters = None

        if fs_customers_snap_extra_filters is not None:
            fs_customers_snap_extra_filters.append(date_filter)
            fs_customers_snap_final_filters = " AND ".join(
                fs_customers_snap_extra_filters
            )
        else:
            fs_customers_snap_final_filters = date_filter

        fs_customers_snap_selected_fields = [
            "cust_code",
            "CUEX_GENERIC_TEXT_03",
            "CUCO_DEDICATED_PROPERTY_03_INTCODE",
            "CUST_LEGAL_PROCESS_STATUS",
            "CUST_LEGAL_PROCESS_STATUS_ACTIVITY_DATE",
            "CUST_CUSTOM_BIT_01",
            "CUST_SLC_ID",
            "CSTM_CUST_INT_03",
            "CUEX_GENERIC_BIT_03",
            "CUST_STAFF_CO",
            "CUST_GENERIC_TEXT_01",
            "CUEX_GENERIC_TEXT_22", ##new
            "CUEX_GENERIC_TEXT_20", ##new
        ]

        fs_customers_snap = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "qca_fs_customers_snap",
            fs_customers_snap_final_filters,
            fs_customers_snap_selected_fields,
        )

        fs_customers_snap = fs_customers_snap.distinct()
        # ===============================fsad_list_of_values===================
        fsad_list_of_values_extra_filters = None

        if fsad_list_of_values_extra_filters is not None:
            fsad_list_of_values_extra_filters.append(date_filter)
            fsad_list_of_values_final_filters = " AND ".join(
                fsad_list_of_values_extra_filters
            )
        else:
            fsad_list_of_values_final_filters = date_filter

        fsad_list_of_values_selected_fields = [
            "LOV_DESC",
            "LOV_CODE",
            "LOVT_CODE",
            "LOVT_DESC",
            "LOV_COMMENTS",
        ]

        fsad_list_of_values = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "qca_fsad_list_of_values",
            fsad_list_of_values_final_filters,
            fsad_list_of_values_selected_fields,
        )

        fsad_list_of_values_1 = fsad_list_of_values
        fsad_list_of_values_2 = fsad_list_of_values_1.withColumnRenamed(
            "LOVT_CODE", "LOVT_CODE_2"
        ).withColumnRenamed("LOV_DESC", "LOV_DESC_2")

        # Occupation description lookup (LOVT_CODE = 141 -> CUST_OCCUPATION).
        # Maps the raw occupation code (cust_occupation == LOV_DESC) to its
        # Greek description held in LOV_COMMENTS. Deduplicate by code to avoid
        # join fan-out, keeping the first non-null description per code.
        occupation_lov = (
            fsad_list_of_values.filter(fsad_list_of_values["LOVT_CODE"] == 141)
            .select(
                fsad_list_of_values["LOV_DESC"].alias("OccupationLovCode"),
                fsad_list_of_values["LOV_COMMENTS"].alias("OccupationDesc"),
            )
            .dropDuplicates(["OccupationLovCode"])
        )

        # ===============================fsad_customers_generic_open===========

        fsad_customers_generic_open_extra_filters = None

        if fsad_customers_generic_open_extra_filters is not None:
            fsad_customers_generic_open_extra_filters.append(date_filter)
            fsad_customers_generic_open_final_filters = " AND ".join(
                fsad_customers_generic_open_extra_filters
            )
        else:
            fsad_customers_generic_open_final_filters = date_filter

        fsad_customers_generic_open_selected_fields = [
            "CUSG_GENERIC_DATETIME_07",
            "CUSG_GENERIC_NVARCHAR_20",
            "CUSG_GENERIC_NVARCHAR_35", #new
            "cust_code",
        ]

        fsad_customers_generic_open = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "qca_fsad_customers_generic_open",
            fsad_customers_generic_open_final_filters,
            fsad_customers_generic_open_selected_fields,
        )

        # ==============================dim_customer_static====================

        dim_customer_static_extra_filters = None

        if dim_customer_static_extra_filters is not None:
            dim_customer_static_extra_filters.append(date_filter)
            dim_customer_static_final_filters = " AND ".join(
                dim_customer_static_extra_filters
            )
        else:
            dim_customer_static_final_filters = date_filter

        dim_customer_static_selected_fields = [
            "cust_id",
            "cust_cif",
            "cust_first_name",
            "cust_last_name",
            "cust_fullname_latin",
            "cust_fath_name",
            "cust_title",
            "cust_sex",
            "cust_mar_status",
            "cust_citizenship",
            "cust_occupation",
            "cust_birthdate",
            "cust_afm",
            "cust_id_num",
            "cust_id_type",
            "cust_annual_income",
            "businessdate",
        ]

        dim_customer_static = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "qca_dim_customer_static",
            dim_customer_static_final_filters,
            dim_customer_static_selected_fields,
        )

        # ============================== fs_cases_snap ====================

        fs_cases_snap_extra_filters = None

        if fs_cases_snap_extra_filters is not None:
            fs_cases_snap_extra_filters.append(date_filter)
            fs_cases_snap_final_filters = " AND ".join(fs_cases_snap_extra_filters)
        else:
            fs_cases_snap_final_filters = date_filter

        fs_cases_snap_selected_fields = ["cust_code"]

        fs_cases_snap = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "qca_fs_cases_snap",
            fs_cases_snap_final_filters,
            fs_cases_snap_selected_fields,
        )

        fs_cases_snap = fs_cases_snap.select("cust_code").distinct()

        # =================== dim_legal_process_status ============
        dim_legal_process_status_extra_filters = None

        if dim_legal_process_status_extra_filters is not None:
            dim_legal_process_status_extra_filters.append(date_filter)
            dim_legal_process_status_final_filters = " AND ".join(
                dim_legal_process_status_extra_filters
            )
        else:
            dim_legal_process_status_final_filters = date_filter

        dim_legal_process_status_selected_fields = ["LEPR_DESC", "LEPR_ID"]

        dim_legal_process_status = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "qca_dim_legal_process_status",
            dim_legal_process_status_final_filters,
            dim_legal_process_status_selected_fields,
        )

        # =============================at_customers========================= #new table
        at_customers_extra_filters = None
   
        if at_customers_extra_filters is not None:
           at_customers_extra_filters.append(date_filter)
           at_customers_final_filters = " AND ".join(
               at_customers_extra_filters
           )
        else:
           at_customers_final_filters = date_filter
   
        at_customers_selected_fields = [
           "cust_code",
           "CUST_GENERIC_TEXT_01",
           "CUST_CITIZENSHIP",
           "CUCO_DEDICATED_PROPERTY_14_INTCODE"
           ]
   
        at_customers = load_landed_delta_table(
           spark,
           catalog_landed,  # type: ignore
           "qca_at_customers",
           at_customers_final_filters,
           at_customers_selected_fields,
        )
   
        at_customers = at_customers.distinct()
 

        # =============================at_customer_extra========================= new table
        at_customer__extra_extra_filters = None
 
        if at_customer__extra_extra_filters is not None:
            at_customer__extra_extra_filters.append(date_filter)
            at_customer_extra_final_filters = " AND ".join(
                at_customer_extra_final_filters
            )
        else:
            at_customer_extra_final_filters = date_filter
 
        at_customer_extra_selected_fields = [
            "cust_code",
            "CUEX_GENERIC_BIT_01", ##new
            "CUEX_GENERIC_TEXT_16",
            "CUEX_GENERIC_TEXT_21",
            ]
 
        at_customer_extra = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "qca_at_customer_extra",
            at_customer_extra_final_filters,
            at_customer_extra_selected_fields,
        )
 
        at_customer_extra = at_customer_extra.distinct()
   

 
        # ================== portfolio ==================

        portfolio_extra_filters = None

        if portfolio_extra_filters is not None:
            portfolio_extra_filters.append(date_filter)
            portfolio_final_filters = " AND ".join(portfolio_extra_filters)
        else:
            portfolio_final_filters = date_filter

        portfolio_selected_fields = [
            "prefix",
            "PortfolioName",
            "portfolio_id_qca",
            "Portfolioid", #new
        ]

        portfolio = load_landed_portfolio_table(  # noqa: F841
            spark,
            curated_zone,  # type: ignore
            "portfolio",
            portfolio_final_filters,
            portfolio_selected_fields,
        )

        # ==============================party_qualco_AB====================

        party_qualco_AB = (
            dim_customer_static.join(
                fs_cases_snap,
                dim_customer_static["cust_id"] == fs_cases_snap["cust_code"],
                "inner",
            )
            .join(
                fs_customers_snap_extra,
                (
                    dim_customer_static["cust_id"]
                    == fs_customers_snap_extra["cust_code"]
                ),
                "left",
            )
            .join(
                fsad_customers_extra_open,
                (
                    dim_customer_static["cust_id"]
                    == fsad_customers_extra_open["cust_code"]
                ),
                "left",
            )
            .join(
                fs_customers_snap,
                (
                    (dim_customer_static["cust_id"] == fs_customers_snap["cust_code"])
                    & (fs_customers_snap.CUST_SLC_ID == "ALL")
                ),
                "left",
            )
            .join(
                fsad_customers_generic_open,
                (
                    dim_customer_static["cust_id"]
                    == fsad_customers_generic_open["cust_code"]
                ),
                "left",
            )
            .join(
                fsad_list_of_values,
                (
                    (
                        fsad_list_of_values["LOV_DESC"]
                        == fs_customers_snap_extra["CUSX_GENERIC_NVARCHAR_01"]
                    )
                    & (fsad_list_of_values["LOVT_CODE"] == 863)
                ),
                "left",
            )
            .join(
                fsad_list_of_values_2,
                (
                    (
                        fsad_list_of_values_2["LOV_CODE"]
                        == fs_customers_snap["CUCO_DEDICATED_PROPERTY_03_INTCODE"]
                    )
                    & (fsad_list_of_values_2["LOVT_CODE_2"] == 834)
                ),
                "left",
            )
            .join(
                at_customers,
                (
                    (fs_customers_snap["cust_code"] == at_customers["cust_code"])  ## new join
                ),
                "left",
            )
            .join(
                portfolio,
                substring(dim_customer_static["cust_cif"], 1, 3) == portfolio["prefix"],
                "left",
            )
            .join(
                dim_legal_process_status,
                dim_legal_process_status.LEPR_ID
                == fs_customers_snap.CUST_LEGAL_PROCESS_STATUS,
                "left",
            )


            .join( ## new
                at_customer_extra,
                at_customer_extra.cust_code
                == fs_customers_snap.cust_code,
                "left",
            )
            .join(  # occupation code -> Greek description
                occupation_lov,
                dim_customer_static["cust_occupation"]
                == occupation_lov["OccupationLovCode"],
                "left",
            )
        )

        # ================ party_qualco_AB ============

        party_qualco_AB_primary_key_fields = [
            dim_customer_static.cust_id,
            lit("qca"),
        ]

        party_qualco_AB = (
            party_qualco_AB.withColumn(
                "PartyID",
                sha2(concat_ws(";", *party_qualco_AB_primary_key_fields), 256),
            )
            .withColumn(
                "CDI",
                dim_customer_static.cust_cif,
            )
            .withColumn(
                "FirstNameGR",
                dim_customer_static["cust_first_name"],
            )
            .withColumn(
                "FirstNameENG",
                dim_customer_static["cust_fullname_latin"],
            )
            .withColumn("LastNameGR", dim_customer_static["cust_last_name"])
            .withColumn(
                "LastNameENG",
                dim_customer_static.cust_fullname_latin,
            )
            .withColumn(
                "FirstFatherNameGR",
                dim_customer_static["cust_fath_name"],
            )
            .withColumn(
                "FirstMotherNameGR",
                fs_customers_snap["CUST_GENERIC_TEXT_01"],
            )
            .withColumn(
                "PrimaryCommercialGR",
                dim_customer_static["cust_fullname_latin"],
            )
            .withColumn(
                "GenderCode",
                dim_customer_static["cust_sex"],
            )
            .withColumn(
                "MaritalStatusCode",
                dim_customer_static["cust_mar_status"],
            )
            .withColumn(
                "Citizenship",
                dim_customer_static["cust_citizenship"],
            )
            .withColumn(
                "OccupationCode",
                dim_customer_static["cust_occupation"],
            )
            .withColumn(
                "PassportNumber",
                when(
                    (
                        dim_customer_static["cust_id_type"].isin(
                            "ΔΙ  ΔΙΑΒΑΤΗΡ.",
                            "ΔΙΑΒΑΤΗΡΙΑ ΛΟΙΠΑ (ΕΚΔ. ΞΕΝΩΝ ΑΡΧΩΝ)",
                            "ΔΙΑΒΑΤΗΡΙΟ",
                            "ΔΙΑΒΑΤΗΡΙΟ (ΕΚΔ. ΕΛΛΗΝΙΚΩΝ ΑΡΧΩΝ ΜΟΝΟ)",
                        )
                    ),
                    dim_customer_static["CUST_ID_NUM"],
                ).otherwise(None),
            )
            .withColumn(
                "PassportIssuingAuthority",
                when(
                    (
                        dim_customer_static["cust_id_type"].isin(
                            "ΔΙ  ΔΙΑΒΑΤΗΡ.",
                            "ΔΙΑΒΑΤΗΡΙΑ ΛΟΙΠΑ (ΕΚΔ. ΞΕΝΩΝ ΑΡΧΩΝ)",
                            "ΔΙΑΒΑΤΗΡΙΟ",
                            "ΔΙΑΒΑΤΗΡΙΟ (ΕΚΔ. ΕΛΛΗΝΙΚΩΝ ΑΡΧΩΝ ΜΟΝΟ)",
                        )
                    ),
                    dim_customer_static["cust_id_type"],
                ).otherwise(None),
            )
            .withColumn(
                "NationalNumber",
                when(
                    (
                        dim_customer_static["cust_id_type"].isin(
                            "ΤΑΥΤΟΤ Ε.Ε.",
                            "ΤΑΥΤΟΤΗΤΑ ΕΝΟΠΛΩΝ ΔΥΝΑΜΕΩΝ (ΓΕΑ) - MONIM",
                            "ΤΑΥΤΟΤΗΤΑ ΕΝΟΠΛΩΝ ΔΥΝΑΜΕΩΝ (ΓΕΑ) - MONIMOY",
                        )
                    ),
                    dim_customer_static["CUST_ID_NUM"],
                ),
            )
            .withColumn(
                    "NationalNumberIssuingAuthority", ## new based on qc values
                    when(
                        (dim_customer_static["cust_id_type"].isin(
                            "ΝΕΑ ΑΣΤΥΝΟΜΙΚΗ ΤΑΥΤΟΤΗΤΑ"
                            ,"ΤΑΥΤΟΤΗΤΑ ΕΝΟΠΛΩΝ ΔΥΝΑΜΕΩΝ (ΓΕΑ) - MONIM"
                            ,"ΤΑΥΤΟΤΗΤΑ ΕΝΟΠΛΩΝ ΔΥΝΑΜΕΩΝ (ΓΕΝ) - MONIM"
                            ,"ΤΑΥΤΟΤΗΤΑ ΕΝΟΠΛΩΝ ΔΥΝΑΜΕΩΝ (ΓΕΣ) - MONIM"
                            ,"ΤΑΥΤΟΤΗΤΑ ΟΜΟΓΕΝΩΝ"
                            ,"ΤΑΥΤΟΤΗΤΑ ΣΩΜΑΤΩΝ ΑΣΦΑΛΕΙΑΣ ΕΛΛΗΝΙΚΗ ΑΣΤ"
                            ,"ΤΑΥΤΟΤΗΤΑ ΣΩΜΑΤΩΝ ΑΣΦΑΛΕΙΑΣ ΛΙΜΕΝΙΚΟ"
                            ,"ΤΑΥΤΟΤΗΤΑ ΣΩΜΑΤΩΝ ΑΣΦΑΛΕΙΑΣ ΠΥΡΟΣΒΕΣΤΙΚΗ"
                        )),
                             dim_customer_static["cust_id_type"],
                    )
                    .otherwise(None)
                    .cast("string"),
                )
            .withColumn(
                "MilitaryNumber",
                when(
                    (dim_customer_static["cust_id_type"].isin("ΕΣ  ΣΤΡΑΤΟΥ")),
                    dim_customer_static["CUST_ID_NUM"],
                ).otherwise(None),
            )
            .withColumn(
                "MilitaryAuthority",
                when(
                    (dim_customer_static["cust_id_type"].isin("ΕΣ  ΣΤΡΑΤΟΥ")),
                    dim_customer_static["cust_id_type"],
                ).otherwise(None),
            )
            .withColumn(
                "SocialSecurityNumber",
                when(
                    (dim_customer_static["cust_id_type"].isin("SocialSecurityNumber")),
                    dim_customer_static["CUST_ID_NUM"],
                ).otherwise(None),
            )
            .withColumn(
                "ChildrenCertification",
                when(
                    (
                        dim_customer_static["cust_id_type"].isin(
                            "ΠΙΣΤ ΑΝΗΛΙΚΟΥ", "ΠΙΣΤΟΠΟΙΗΤΙΚΟ ΑΝΗΛΙΚΟΥ"
                        )
                    ),
                    dim_customer_static["CUST_ID_NUM"],
                ).otherwise(None),
            )
            .withColumn(
                "ChildrenCertificationIssuingAuthority",
                when(
                    (
                        dim_customer_static["cust_id_type"].isin(
                            "ΠΙΣΤ ΑΝΗΛΙΚΟΥ", "ΠΙΣΤΟΠΟΙΗΤΙΚΟ ΑΝΗΛΙΚΟΥ"
                        )
                    ),
                    dim_customer_static["cust_id_type"],
                ).otherwise(None),
            )
            .withColumn(
                "MonitoringUnitCode",
                fsad_customers_extra_open["CUEX_GENERIC_TEXT_02"],
            )
            .withColumn(
                "PartyType",
                when(
                    (fs_customers_snap_extra["cusx_generic_bit_01"] == lit(False)),
                    "Individual",
                )
                .when(
                    (fs_customers_snap_extra["cusx_generic_bit_01"] == lit(True)),
                    "Organization",
                )
                .otherwise("Both"),
            )
            .withColumn(
                "BalanceCategory",
                fs_customers_snap["CUEX_GENERIC_TEXT_22"], ##new
                )
            .withColumn(
                "ExtraSegment",
                fs_customers_snap["CUEX_GENERIC_TEXT_20"], ##new
                )
            .withColumn(
                "LastMotherNameGR",
                at_customers["CUST_GENERIC_TEXT_01"],
                )
            .withColumn(
                "PartyBankruptcyStage",
                at_customers["CUCO_DEDICATED_PROPERTY_14_INTCODE"], ##new
                )
            .withColumn(
                "NationalityCode",
                at_customers["CUST_CITIZENSHIP"],
                )
            .withColumn( #new
                "PartyStatusCode",
                at_customer_extra["CUEX_GENERIC_BIT_01"],
                )
            .withColumn( #new
                "PartyStatus",
                when(
                        at_customer_extra["CUEX_GENERIC_BIT_01"] == lit(True), "deceased"
                    ).otherwise("active"),
                )
    

            .withColumn(  
                "SubServicer",
                fsad_customers_generic_open["CUSG_GENERIC_NVARCHAR_35"], ##new
                )
            .withColumn( #new
                "Segment",
                at_customer_extra["CUEX_GENERIC_TEXT_16"],
                )
            .withColumn( #new
                "SubSegment",
                at_customer_extra["CUEX_GENERIC_TEXT_21"],
                )
            )

        # ================== Selection of Final Columns ==================

        party_qualco_AB = party_qualco_AB.select(
            party_qualco_AB.PartyID.alias("PartyID"),
            lit("qca").cast("string").alias("SourceSystem"),
            coalesce(portfolio.portfolio_id_qca, lit(2))
            .cast("string")
            .alias("PortfolioKey"),
            coalesce(portfolio.PortfolioName, lit("Core Retail"))
            .cast("string")
            .alias("PortfolioName"),            
            lit(from_utc_timestamp(current_timestamp(), "Europe/Athens"))
            .cast("timestamp")
            .alias("IngestionTimestamp"),
            lit(None).cast("string").alias("BatchId"),
            party_qualco_AB.CDI.cast("string").alias("CDI"),
            lit(None).cast("string").alias("RepresentedByPartyKey"),
            lit(None).cast("string").alias("RepresentedByPartyName"),
            lit(None).cast("string").alias("RepresentedByPartyType"),
            dim_customer_static.cust_id.cast("string").alias("PartyKey"),
            dim_customer_static.cust_id.cast("string").alias("PartyRepresentKey"),
            fs_customers_snap_extra.cusx_generic_bit_01.cast("string").alias(
                "PartyTypeCode"
            ),
            party_qualco_AB.PartyType.cast("string").alias("PartyType"),
            party_qualco_AB.PartyStatusCode.cast("string").alias("PartyStatusCode"), #new
            party_qualco_AB.PartyStatus.cast("string").alias("PartyStatus"),#new but needs confirmation to be sure
            lit(None).cast("date").alias("PartyStatusDate"),
            fs_customers_snap_extra.CUSX_GENERIC_NVARCHAR_01.cast("string").alias(
                "L3869StatusCode"
            ),
            fs_customers_snap_extra.CUSX_GENERIC_NVARCHAR_01.cast("string").alias(
                "L3869Status"
            ),
            fsad_customers_generic_open.CUSG_GENERIC_DATETIME_07.cast(
                "timestamp"
            ).alias("L3869CourtDate"),
            lit(None).alias("OCWStatus"),
            fs_customers_snap_extra.CUSX_GENERIC_BIT_02.cast("boolean").alias(
                "DeceasedFlag"
            ),
            fs_customers_snap.CUCO_DEDICATED_PROPERTY_03_INTCODE.cast("int").alias(
                "L3869InterbankingStatusCode"
            ),
            fsad_list_of_values_2.LOV_DESC_2.cast("string").alias(
                "L3869InterbankingStatus"
            ),

            fs_customers_snap.CUEX_GENERIC_TEXT_03.alias("Watchlist"),
            party_qualco_AB.FirstNameGR.cast("string").alias("FirstNameGR"),
            party_qualco_AB.FirstNameENG.cast("string").alias("FirstNameENG"),
            party_qualco_AB.LastNameGR.cast("string").alias("LastNameGR"),
            party_qualco_AB.LastNameENG.cast("string").alias("LastNameENG"),
            party_qualco_AB.FirstFatherNameGR.cast("string").alias("FirstFatherNameGR"),
            lit(None).cast("string").alias("FirstFatherNameENG"),
            lit(None).cast("string").alias("LastFatherNameGR"),
            lit(None).cast("string").alias("LastFatherNameENG"),
            party_qualco_AB.FirstMotherNameGR.cast("string").alias("FirstMotherNameGR"),
            party_qualco_AB.LastMotherNameGR.cast("string").alias("LastMotherNameGR"), #new
            lit(None).cast("string").alias("LastMotherNameENG"),
            lit(None).cast("string").alias("LegalNameGR"),
            lit(None).cast("string").alias("LegalNameENG"),
            party_qualco_AB.PrimaryCommercialGR.cast("string").alias(
                "PrimaryCommercialGR"
            ),
            lit(None).cast("string").alias("PrimaryCommercialENG"),
            dim_customer_static.cust_title.cast("string").alias("SalutationTypeCode"),
            lit(None).cast("string").alias("SalutationType"),
            party_qualco_AB.GenderCode.cast("string").alias("GenderCode"),
            lit(None).cast("string").alias("Gender"),
            party_qualco_AB.MaritalStatusCode.cast("string").alias("MaritalStatusCode"),
            party_qualco_AB.MaritalStatusCode.cast("string").alias("MaritalStatus"),
            party_qualco_AB.Citizenship.cast("string").alias("Citizenship"),
            party_qualco_AB.Citizenship.cast("string").alias("CitizenshipCode"),
            party_qualco_AB.NationalityCode.cast("string").alias("NationalityCode"), #new
            party_qualco_AB.NationalityCode.cast("string").alias("Nationality"), #new
            party_qualco_AB.OccupationCode.cast("string").alias("OccupationCode"),
            coalesce(
                occupation_lov.OccupationDesc, party_qualco_AB.OccupationCode
            )
            .cast("string")
            .alias("Occupation"),
            dim_customer_static.cust_birthdate.cast("string").alias("BirthDate"),
            lit(None).cast("string").alias("BirthPlace"),
            lit(None).cast("date").alias("DeathDate"),
            lit(None).cast("string").alias("CommunicationLanguage"),
            lit(None).cast("int").alias("CommunicationLanguageCode"),
            dim_customer_static.cust_afm.cast("string").alias("TaxRegistrationNumber"),
            lit(None).cast("string").alias("TaxResidenceCountry"),
            lit(None).cast("string").alias("OrganizationOriginCountry"),
            party_qualco_AB.PassportNumber.cast("string").alias("PassportNumber"),
            party_qualco_AB.PassportIssuingAuthority.cast("string").alias(
                "PassportIssuingAuthority"
            ),
            lit(None).cast("date").alias("PassportIssueDate"),
            lit(None).cast("date").alias("PassportExpiredDate"),
            party_qualco_AB.NationalNumber.cast("string").alias("NationalNumber"),
            party_qualco_AB.NationalNumberIssuingAuthority.cast("string").alias(
                "NationalNumberIssuingAuthority"
            ),
            lit(None).cast("date").alias("NationalNumberIssueDate"),
            lit(None).cast("date").alias("NationalNumberExpiredDate"),
            party_qualco_AB.MilitaryNumber.cast("string").alias("MilitaryNumber"),
            party_qualco_AB.MilitaryAuthority.cast("string").alias("MilitaryAuthority"),
            lit(None).cast("date").alias("MilitaryIssueDate"),
            lit(None).cast("date").alias("MilitaryExpiredDate"),
            party_qualco_AB.SocialSecurityNumber.cast("string").alias(
                "SocialSecurityNumber"
            ),
            party_qualco_AB.ChildrenCertification.cast("string").alias(
                "ChildrenCertification"
            ),
            party_qualco_AB.ChildrenCertificationIssuingAuthority.cast("string").alias(
                "ChildrenCertificationIssuingAuthority"    # but not decommission related, the column was created but wasnt used
            ),
            lit(None).cast("string").alias("EnterpriseStartDate"),
            lit(None).cast("date").alias("EnterpriseEndDate"),
            lit(None).cast("int").alias("EconomicActivityCode"),
            lit(None).cast("string").alias("EconomicActivity"),
            lit(None).cast("int").alias("BranchActivityCode"),
            lit(None).cast("string").alias("BranchActivity"),
            lit(None).cast("string").alias("NumberOfChildren"),
            lit(None).cast("string").alias("GeneralCommercialRegister"),
            lit(None).cast("string").alias("NumberofEmployees"),
            dim_customer_static.cust_annual_income.cast("string").alias("TotalIncome"),
            lit(None).cast("string").alias("BalanceSheetTotalAmount"),
            lit(None).cast("string").alias("AnnualTurnoverAmount"),
            lit(None).cast("string").alias("GroupCode"),
            lit(None).cast("string").alias("GroupName"),
            lit(None).alias("GroupCodePortfolio"),
            party_qualco_AB.PartyBankruptcyStage.cast("int").alias("PartyBankruptcyStage"),
            lit(None).cast("string").alias("CovidFlag"),
            lit(None).cast("int").alias("CovidCategoryCode"),
            lit(None).cast("string").alias("CovidCategory"),
            lit(None).cast("int").alias("AffectedCategoryCode"),
            lit(None).cast("string").alias("AffectedCategory"),
            party_qualco_AB.MonitoringUnitCode.cast("string").alias(
                "MonitoringUnitCode"
            ),
            party_qualco_AB.MonitoringUnitCode.cast("string").alias("MonitoringUnit"),
            lit(None).cast("int").alias("CreditRatingTypeCode"),
            lit(None).cast("string").alias("CreditRatingType"),
            lit(None).cast("int").alias("EducationLevelCode"),
            lit(None).cast("string").alias("EducationLevel"),
            party_qualco_AB.BalanceCategory.cast("string").alias("BalanceCategoryCode"), #new
            party_qualco_AB.BalanceCategory.cast("string").alias("BalanceCategory"),#new
            party_qualco_AB.ExtraSegment.alias("ExtraSegmentCode"),#new
            party_qualco_AB.ExtraSegment.alias("ExtraSegment"),#new
            party_qualco_AB.Segment.alias("SegmentCode"),
            party_qualco_AB.Segment.alias("Segment"),
            party_qualco_AB.SubSegment.alias("SubSegmentCode"),
            party_qualco_AB.SubSegment.alias("SubSegment"),
            lit(None).cast("int").alias("LegalFormCode"),
            lit(None).cast("string").alias("LegalForm"),
            party_qualco_AB.CUST_CUSTOM_BIT_01.alias("IsWholeshaleFlag"),
            lit(None).cast("string").alias("EmploymentStatus"),
            lit(None).alias("EmploymentStatusCode"),
            lit(None).cast("date").alias("EmploymentStatusDate"),
            lit(None).cast("string").alias("CreatedBy"),
            lit(None).cast("timestamp").alias("CreatedByDate"),
            lit(None).cast("string").alias("LastUpdatedBy"),
            lit(None).cast("timestamp").alias("LastUpdatedByDate"),
            party_qualco_AB.CUSG_GENERIC_NVARCHAR_20.alias("ResubmissionStatus"),
            party_qualco_AB.CSTM_CUST_INT_03.alias("N4605Flag"),
            party_qualco_AB.CUEX_GENERIC_BIT_03.alias("ConnectedBorrowerFlag"),
            party_qualco_AB.CUST_STAFF_CO.alias("StaffFlag"),
            party_qualco_AB.CUSX_GENERIC_BIT_05.cast("boolean").alias(
                "L3869BorrowerHasOver50KBalance"
            ),
            lit(None).alias("TargetLegalActionCode"),
            lit(None).alias("TargetLegalAction"),
            party_qualco_AB.SubServicer.cast("string").alias("SubServicer"), #new
            lit(None).cast("string").alias("RiskDescription"),
            lit(None).cast("string").alias("MobileTel"),
            lit(None).cast("string").alias("MobileTel2"),
            party_qualco_AB.businessdate.cast("date").alias("BusinessDate"),
        ).distinct() #new
        # ================== Append ==================

        party_qualco_AB.write.format("delta").clusterBy("BusinessDate").mode(
            "append"
        ).saveAsTable(curated_table_path)

    else:
        null_args = [
            arg_name
            for arg_name, arg in args
            if arg
             in [
                curated_zone,
                catalog_landed,
                reference_zone,
                run_date,
                schema,
                env_prefix,
            ]
             and arg is None
        ]

        for arg_name in null_args:
            logging.error(f"No {arg_name} specified!")

    logging.info("We did it!")


In [0]:
load_curated_party_qca(spark, curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix)